# ORIZZONTE — Ingestion corpus con MinerU (GPU)

Parsa tutti i PDF del corpus in Markdown + metadati, pronto per `MarkdownHeaderTextSplitter`.

**Setup richiesto:**
1. Carica la cartella `data/raw` (con le sottocartelle TOPIC) come dataset Kaggle, es. `orizzonte-corpus-raw`
2. Aggiungi il dataset a questo notebook (Add Input)
3. Settings → Accelerator: **GPU T4 x2** (ne basta una) · Internet: **ON**
4. Esegui tutto; alla fine scarica `corpus_processed.zip` dall'output

In [ ]:
# ── 1. Verifica GPU ──
import torch
print("CUDA disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# ── 2. Installazione MinerU (backend pipeline, adatto alla T4) ──
%pip install -q "mineru[core]"
!mineru --version

In [ ]:
# ── 3. Configurazione percorsi ──
from pathlib import Path

# ADATTA QUESTO allo slug del tuo dataset:
RAW_DIR = Path("/kaggle/input/orizzonte-corpus-raw")
OUT_DIR = Path("/kaggle/working/processed")
TMP_DIR = Path("/kaggle/working/_mineru_tmp")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(RAW_DIR.rglob("*.pdf"))
print(f"{len(pdfs)} PDF trovati")
for p in pdfs[:5]:
    print("  ", p.relative_to(RAW_DIR))

In [ ]:
# ── 4. Ingestion: MinerU su ogni PDF, con metadati topic/livelli ──
import json, re, shutil, subprocess, time
from datetime import datetime

LEVEL_PREFIX_RE = re.compile(r"^([ABCD](?:\s*,\s*[ABCD])*)\s*-\s*")

def extract_metadata(pdf: Path) -> dict:
    topic = pdf.parent.name if pdf.parent != RAW_DIR else "SENZA_TOPIC"
    m = LEVEL_PREFIX_RE.match(pdf.stem)
    levels = [x.strip() for x in m.group(1).split(",")] if m else []
    title = LEVEL_PREFIX_RE.sub("", pdf.stem).strip()
    return {"topic": topic, "levels": levels, "title": title, "source_pdf": pdf.name}

def parse_with_mineru(pdf: Path) -> str:
    out = TMP_DIR / pdf.stem
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    proc = subprocess.run(["mineru", "-p", str(pdf), "-o", str(out)],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"mineru exit {proc.returncode}: {proc.stderr.strip()[-300:]}")
    md_files = sorted(out.rglob("*.md"))
    if not md_files:
        raise RuntimeError("nessun .md prodotto")
    best = max(md_files, key=lambda f: f.stat().st_size)
    return best.read_text(encoding="utf-8")

manifest, t0 = [], time.time()
for i, pdf in enumerate(pdfs, 1):
    meta = extract_metadata(pdf)
    rel_dir = pdf.parent.relative_to(RAW_DIR)
    dest = OUT_DIR / rel_dir
    dest.mkdir(parents=True, exist_ok=True)
    md_path = dest / (pdf.stem + ".md")
    meta_path = dest / (pdf.stem + ".meta.json")

    if md_path.exists():
        print(f"[{i:>2}/{len(pdfs)}] ↷ già fatto: {pdf.name}")
        manifest.append({**meta, "status": "skipped"})
        continue

    print(f"[{i:>2}/{len(pdfs)}] ⚙ {pdf.name}", flush=True)
    t = time.time()
    try:
        md = parse_with_mineru(pdf)
        md_path.write_text(md, encoding="utf-8")
        n_headers = len(re.findall(r"^#{1,6}\\s", md, flags=re.M))
        meta_path.write_text(json.dumps({**meta, "parser": "mineru",
            "chars": len(md), "headers": n_headers,
            "processed_at": datetime.now().isoformat(timespec="seconds")},
            ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"      ✓ {len(md)//1000} kchar, {n_headers} header — {time.time()-t:.0f}s")
        manifest.append({**meta, "status": "ok"})
    except Exception as e:
        print(f"      ✗ {type(e).__name__}: {e}")
        manifest.append({**meta, "status": f"failed: {type(e).__name__}"})

(OUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
ok = sum(1 for m in manifest if m["status"] == "ok")
print(f"\nFinito in {(time.time()-t0)/60:.0f} min — ok: {ok}, "
      f"falliti: {sum(1 for m in manifest if m['status'].startswith('failed'))}")

In [ ]:
# ── 5. Zip dei risultati (solo .md + .meta.json + manifest) ──
import shutil
shutil.make_archive("/kaggle/working/corpus_processed", "zip", OUT_DIR)
!ls -lh /kaggle/working/corpus_processed.zip
print("\nScarica corpus_processed.zip dal pannello Output e scompattalo in data/processed/ nel progetto locale.")